# NBA Prediction Pipeline Demo

This notebook demonstrates the refactored Python package workflow for the NBA prediction project. It uses the same package entry points as the command line interface:

1. download or reuse raw NBA team game logs;
2. build the processed design matrix;
3. evaluate a baseline model;
4. inspect generated metrics and predictions.

Raw data and generated outputs are intentionally ignored by Git. They are produced through the pipeline commands instead of being committed to the repository.

## Setup

In [ ]:
from pathlib import Path
import json
import sys

import pandas as pd

repo_root = Path.cwd()
if not (repo_root / "src").exists() and (repo_root.parent / "src").exists():
    repo_root = repo_root.parent

sys.path.insert(0, str(repo_root / "src"))

from nba_predict.config import ProjectPaths
from nba_predict.pipeline import NBAPredictionPipeline

paths = ProjectPaths(root=repo_root)
pipeline = NBAPredictionPipeline(paths=paths, cv_folds=3)
paths.ensure_output_dirs()

print(f"Repository root: {paths.root}")
print(f"Raw data path: {paths.raw_data}")
print(f"Design matrix path: {paths.design_matrix}")

## Download Or Reuse Raw Data

If `data/raw/2012_2023_Data.csv` already exists, this notebook reuses it. If it is missing, the next cell downloads team-level NBA game logs for the same season range used by the original R workflow.

Set `RUN_DOWNLOAD = False` before running the cell if you want to avoid external API calls.

In [ ]:
RUN_DOWNLOAD = not paths.raw_data.exists()

if RUN_DOWNLOAD:
    raw_data = pipeline.download_data(
        output_path=paths.raw_data,
        season_start=2013,
        season_end=2023,
    )
    print(f"Downloaded raw data: {raw_data.shape[0]:,} rows")
else:
    raw_data = pd.read_csv(paths.raw_data)
    print(f"Using existing raw data: {raw_data.shape[0]:,} rows")

raw_data.head()

## Build The Design Matrix

The design matrix follows the original project idea: each row is a home-team minus away-team feature comparison based on rolling team performance.

In [ ]:
design_matrix = pipeline.prepare_data(
    raw_path=paths.raw_data,
    output_path=paths.design_matrix,
)

print(f"Design matrix shape: {design_matrix.shape}")
design_matrix.head()

## Evaluate A Baseline Model

This cell evaluates one translated baseline model to keep the demo quick. Set `RUN_ALL_BASELINES = True` to run logistic, inference-logistic, ridge, and lasso together.

In [ ]:
RUN_ALL_BASELINES = False

if RUN_ALL_BASELINES:
    results = pipeline.run_baseline(season="2022-23")
else:
    results = [pipeline.evaluate_model("logistic", season="2022-23")]

summary_rows = []
for result in results:
    metrics = result["metrics"]
    summary_rows.append(
        {
            "model": result["model"],
            "accuracy": metrics["accuracy"],
            "precision": metrics["precision"],
            "recall": metrics["recall"],
            "metrics_path": result["metrics_path"],
            "predictions_path": result["predictions_path"],
        }
    )

pd.DataFrame(summary_rows)

## Inspect Saved Outputs

In [ ]:
metrics_files = sorted(paths.metrics_dir.glob("*_2022-23_metrics.json"))
prediction_files = sorted(paths.predictions_dir.glob("*_2022-23_predictions.csv"))

print("Metrics files:")
for path in metrics_files:
    print(f"- {path.relative_to(paths.root)}")

print("\nPrediction files:")
for path in prediction_files:
    print(f"- {path.relative_to(paths.root)}")

In [ ]:
latest_metrics_path = paths.metrics_dir / "logistic_2022-23_metrics.json"
latest_predictions_path = paths.predictions_dir / "logistic_2022-23_predictions.csv"

metrics = json.loads(latest_metrics_path.read_text())
print(json.dumps({k: v for k, v in metrics.items() if k != "cumulative_errors"}, indent=2))

pd.read_csv(latest_predictions_path).head()